# Medical Image Classification with Transfer Learning and Ensemble Methods

This notebook demonstrates an end-to-end medical image classification pipeline using:
- **Dataset**: DermaMNIST (dermatoscopic images of pigmented skin lesions)
- **Transfer Learning**: VGG-19, ResNet50, GoogLeNet (Inception V1), InceptionV3
- **Ensemble Methods**: Averaging and Voting strategies

## Table of Contents
1. [Package Installation](#1-package-installation)
2. [Imports](#2-imports)
3. [Configuration & Device Setup](#3-configuration--device-setup)
4. [Data Loading & Exploration](#4-data-loading--exploration)
5. [Data Visualization](#5-data-visualization)
6. [Data Preprocessing & Augmentation](#6-data-preprocessing--augmentation)
7. [Model Definitions](#7-model-definitions)
8. [Loss Function & Optimizers](#8-loss-function--optimizers)
9. [Training & Validation Loops](#9-training--validation-loops)
10. [Model Training](#10-model-training)
11. [Learning Curves](#11-learning-curves)
12. [Ensemble Methods](#12-ensemble-methods)
13. [Model Evaluation](#13-model-evaluation)
14. [Results Comparison](#14-results-comparison)
15. [Conclusion](#15-conclusion)

---
## 1. Package Installation

Install all required packages for the medical image classification pipeline.

In [ ]:
# Install required packages
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install medmnist
!pip install matplotlib seaborn
!pip install scikit-learn
!pip install tqdm
!pip install pandas numpy
!pip install pillow

---
## 2. Imports

Import all necessary libraries for data handling, visualization, model building, and evaluation.

In [ ]:
# Core libraries
import os
import random
import numpy as np
import pandas as pd
from collections import defaultdict
import copy
import warnings
warnings.filterwarnings('ignore')

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F

# Torchvision
import torchvision
from torchvision import transforms, models
from torchvision.models import (
    vgg19, VGG19_Weights,
    resnet50, ResNet50_Weights,
    googlenet, GoogLeNet_Weights,
    inception_v3, Inception_V3_Weights
)

# MedMNIST
import medmnist
from medmnist import INFO, Evaluator

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# Metrics
from sklearn.metrics import (
    confusion_matrix, classification_report, 
    accuracy_score, precision_recall_fscore_support,
    roc_auc_score, roc_curve, auc
)
from sklearn.preprocessing import label_binarize

# Progress bar
from tqdm.notebook import tqdm

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

print(f"PyTorch Version: {torch.__version__}")
print(f"Torchvision Version: {torchvision.__version__}")
print(f"MedMNIST Version: {medmnist.__version__}")

---
## 3. Configuration & Device Setup

Set up configuration parameters and check for GPU availability.

In [ ]:
# Configuration
class Config:
    # Dataset
    DATA_FLAG = 'dermamnist'  # Using DermaMNIST dataset
    DOWNLOAD = True
    
    # Training parameters
    BATCH_SIZE = 32
    NUM_EPOCHS = 25
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY = 1e-5
    
    # Image parameters
    IMG_SIZE = 224  # Standard size for pretrained models
    
    # Random seed for reproducibility
    SEED = 42
    
    # Model checkpoints directory
    CHECKPOINT_DIR = './checkpoints'
    
# Set random seeds for reproducibility
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(Config.SEED)

# Create checkpoint directory
os.makedirs(Config.CHECKPOINT_DIR, exist_ok=True)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

---
## 4. Data Loading & Exploration

Load the DermaMNIST dataset and explore its properties.

**DermaMNIST** is based on HAM10000 dataset consisting of 10,015 dermatoscopic images of pigmented skin lesions categorized into 7 classes:
- 0: Actinic keratoses (akiec)
- 1: Basal cell carcinoma (bcc)
- 2: Benign keratosis-like lesions (bkl)
- 3: Dermatofibroma (df)
- 4: Melanoma (mel)
- 5: Melanocytic nevi (nv)
- 6: Vascular lesions (vasc)

In [ ]:
# Get dataset information
info = INFO[Config.DATA_FLAG]
task = info['task']
n_channels = info['n_channels']
n_classes = len(info['label'])
class_names = list(info['label'].values())

print("=" * 60)
print(f"Dataset: {Config.DATA_FLAG.upper()}")
print("=" * 60)
print(f"Task: {task}")
print(f"Number of channels: {n_channels}")
print(f"Number of classes: {n_classes}")
print(f"\nClass names:")
for idx, name in enumerate(class_names):
    print(f"  {idx}: {name}")

In [ ]:
# Load the dataset
DataClass = getattr(medmnist, info['python_class'])

# Load raw data without transforms first for exploration
train_dataset_raw = DataClass(split='train', download=Config.DOWNLOAD)
val_dataset_raw = DataClass(split='val', download=Config.DOWNLOAD)
test_dataset_raw = DataClass(split='test', download=Config.DOWNLOAD)

print(f"\nDataset sizes:")
print(f"  Training set: {len(train_dataset_raw)} samples")
print(f"  Validation set: {len(val_dataset_raw)} samples")
print(f"  Test set: {len(test_dataset_raw)} samples")
print(f"  Total: {len(train_dataset_raw) + len(val_dataset_raw) + len(test_dataset_raw)} samples")

In [ ]:
# Explore a single sample
sample_img, sample_label = train_dataset_raw[0]
print(f"\nSample image:")
print(f"  Type: {type(sample_img)}")
print(f"  Size: {sample_img.size}")
print(f"  Mode: {sample_img.mode}")
print(f"  Label: {sample_label} ({class_names[sample_label[0]]})")

In [ ]:
# Analyze class distribution
def get_class_distribution(dataset):
    labels = []
    for _, label in dataset:
        labels.append(label[0])
    return np.bincount(labels, minlength=n_classes)

train_dist = get_class_distribution(train_dataset_raw)
val_dist = get_class_distribution(val_dataset_raw)
test_dist = get_class_distribution(test_dataset_raw)

# Create distribution dataframe
dist_df = pd.DataFrame({
    'Class': class_names,
    'Train': train_dist,
    'Validation': val_dist,
    'Test': test_dist
})
dist_df['Total'] = dist_df['Train'] + dist_df['Validation'] + dist_df['Test']
dist_df['Train %'] = (dist_df['Train'] / dist_df['Train'].sum() * 100).round(2)

print("\nClass Distribution:")
print(dist_df.to_string(index=False))

---
## 5. Data Visualization

Visualize samples from each class and the class distribution.

In [ ]:
# Visualize class distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Training set distribution
colors = sns.color_palette("husl", n_classes)
axes[0].bar(range(n_classes), train_dist, color=colors)
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Count')
axes[0].set_title('Training Set Distribution')
axes[0].set_xticks(range(n_classes))
axes[0].set_xticklabels([f'{i}' for i in range(n_classes)], rotation=45)

# Validation set distribution
axes[1].bar(range(n_classes), val_dist, color=colors)
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Count')
axes[1].set_title('Validation Set Distribution')
axes[1].set_xticks(range(n_classes))
axes[1].set_xticklabels([f'{i}' for i in range(n_classes)], rotation=45)

# Test set distribution
axes[2].bar(range(n_classes), test_dist, color=colors)
axes[2].set_xlabel('Class')
axes[2].set_ylabel('Count')
axes[2].set_title('Test Set Distribution')
axes[2].set_xticks(range(n_classes))
axes[2].set_xticklabels([f'{i}' for i in range(n_classes)], rotation=45)

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Visualize sample images from each class
def visualize_samples_per_class(dataset, class_names, n_samples=5):
    """Visualize n_samples from each class."""
    n_classes = len(class_names)
    fig, axes = plt.subplots(n_classes, n_samples, figsize=(n_samples * 2, n_classes * 2))
    
    # Collect samples per class
    class_samples = defaultdict(list)
    for img, label in dataset:
        label_idx = label[0]
        if len(class_samples[label_idx]) < n_samples:
            class_samples[label_idx].append(img)
        if all(len(v) >= n_samples for v in class_samples.values()):
            break
    
    for class_idx in range(n_classes):
        for sample_idx in range(n_samples):
            ax = axes[class_idx, sample_idx]
            if sample_idx < len(class_samples[class_idx]):
                img = class_samples[class_idx][sample_idx]
                ax.imshow(img)
            ax.axis('off')
            if sample_idx == 0:
                ax.set_ylabel(f'{class_idx}: {class_names[class_idx][:10]}', 
                             fontsize=10, rotation=0, ha='right', va='center')
    
    plt.suptitle('Sample Images from Each Class', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.savefig('sample_images.png', dpi=150, bbox_inches='tight')
    plt.show()

visualize_samples_per_class(train_dataset_raw, class_names, n_samples=5)

In [ ]:
# Visualize random samples from training set
def visualize_random_samples(dataset, class_names, n_rows=4, n_cols=6):
    """Visualize random samples from the dataset."""
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 2, n_rows * 2.5))
    
    indices = random.sample(range(len(dataset)), n_rows * n_cols)
    
    for idx, ax_idx in enumerate(range(n_rows * n_cols)):
        ax = axes[ax_idx // n_cols, ax_idx % n_cols]
        img, label = dataset[indices[idx]]
        ax.imshow(img)
        ax.set_title(f'{class_names[label[0]][:12]}', fontsize=9)
        ax.axis('off')
    
    plt.suptitle('Random Samples from Training Set', fontsize=14)
    plt.tight_layout()
    plt.savefig('random_samples.png', dpi=150, bbox_inches='tight')
    plt.show()

visualize_random_samples(train_dataset_raw, class_names)

---
## 6. Data Preprocessing & Augmentation

Define transforms for training (with augmentation) and validation/test sets.

In [ ]:
# ImageNet statistics for normalization (used by pretrained models)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# Training transforms with data augmentation
train_transform = transforms.Compose([
    transforms.Resize((Config.IMG_SIZE, Config.IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

# Validation/Test transforms (no augmentation)
val_transform = transforms.Compose([
    transforms.Resize((Config.IMG_SIZE, Config.IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

# Special transform for InceptionV3 (requires 299x299 input)
train_transform_inception = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

val_transform_inception = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

print("Transforms defined successfully!")

In [ ]:
# Create datasets with transforms
def create_datasets(transform_train, transform_val):
    """Create train, validation, and test datasets with specified transforms."""
    train_ds = DataClass(split='train', transform=transform_train, download=Config.DOWNLOAD)
    val_ds = DataClass(split='val', transform=transform_val, download=Config.DOWNLOAD)
    test_ds = DataClass(split='test', transform=transform_val, download=Config.DOWNLOAD)
    return train_ds, val_ds, test_ds

# Standard datasets (for VGG19, ResNet50, GoogLeNet)
train_dataset, val_dataset, test_dataset = create_datasets(train_transform, val_transform)

# Inception datasets (299x299)
train_dataset_inception, val_dataset_inception, test_dataset_inception = create_datasets(
    train_transform_inception, val_transform_inception
)

print("Datasets created with transforms!")

In [ ]:
# Create data loaders
def create_dataloaders(train_ds, val_ds, test_ds, batch_size=Config.BATCH_SIZE):
    """Create data loaders for train, validation, and test sets."""
    train_loader = DataLoader(
        train_ds, 
        batch_size=batch_size, 
        shuffle=True, 
        num_workers=2,
        pin_memory=True
    )
    val_loader = DataLoader(
        val_ds, 
        batch_size=batch_size, 
        shuffle=False, 
        num_workers=2,
        pin_memory=True
    )
    test_loader = DataLoader(
        test_ds, 
        batch_size=batch_size, 
        shuffle=False, 
        num_workers=2,
        pin_memory=True
    )
    return train_loader, val_loader, test_loader

# Standard loaders
train_loader, val_loader, test_loader = create_dataloaders(
    train_dataset, val_dataset, test_dataset
)

# Inception loaders
train_loader_inception, val_loader_inception, test_loader_inception = create_dataloaders(
    train_dataset_inception, val_dataset_inception, test_dataset_inception
)

print(f"Data loaders created!")
print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

In [ ]:
# Visualize augmented images
def visualize_augmentations(dataset_raw, transform, n_augmentations=5):
    """Visualize the effect of augmentations on a single image."""
    # Get a sample image
    img, label = dataset_raw[0]
    
    fig, axes = plt.subplots(1, n_augmentations + 1, figsize=((n_augmentations + 1) * 2.5, 3))
    
    # Original image
    axes[0].imshow(img)
    axes[0].set_title('Original')
    axes[0].axis('off')
    
    # Augmented images
    for i in range(n_augmentations):
        aug_img = transform(img)
        # Denormalize for visualization
        aug_img_np = aug_img.numpy().transpose(1, 2, 0)
        aug_img_np = aug_img_np * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
        aug_img_np = np.clip(aug_img_np, 0, 1)
        
        axes[i + 1].imshow(aug_img_np)
        axes[i + 1].set_title(f'Augmented {i + 1}')
        axes[i + 1].axis('off')
    
    plt.suptitle(f'Data Augmentation Examples (Class: {class_names[label[0]]})', fontsize=12)
    plt.tight_layout()
    plt.savefig('augmentation_examples.png', dpi=150, bbox_inches='tight')
    plt.show()

visualize_augmentations(train_dataset_raw, train_transform)

---
## 7. Model Definitions

Define pretrained models (VGG-19, ResNet50, GoogLeNet, InceptionV3) with modified final layers for our classification task.

In [ ]:
class TransferLearningModel(nn.Module):
    """Wrapper class for transfer learning models."""
    
    def __init__(self, model_name, num_classes, pretrained=True, freeze_backbone=False):
        super(TransferLearningModel, self).__init__()
        self.model_name = model_name
        
        if model_name == 'vgg19':
            weights = VGG19_Weights.IMAGENET1K_V1 if pretrained else None
            self.model = vgg19(weights=weights)
            # Modify classifier
            in_features = self.model.classifier[6].in_features
            self.model.classifier[6] = nn.Linear(in_features, num_classes)
            
        elif model_name == 'resnet50':
            weights = ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
            self.model = resnet50(weights=weights)
            # Modify final fully connected layer
            in_features = self.model.fc.in_features
            self.model.fc = nn.Linear(in_features, num_classes)
            
        elif model_name == 'googlenet':
            weights = GoogLeNet_Weights.IMAGENET1K_V1 if pretrained else None
            self.model = googlenet(weights=weights, aux_logits=True)
            # Modify main classifier
            in_features = self.model.fc.in_features
            self.model.fc = nn.Linear(in_features, num_classes)
            # Modify auxiliary classifiers
            if self.model.aux_logits:
                self.model.aux1.fc2 = nn.Linear(1024, num_classes)
                self.model.aux2.fc2 = nn.Linear(1024, num_classes)
            
        elif model_name == 'inception_v3':
            weights = Inception_V3_Weights.IMAGENET1K_V1 if pretrained else None
            self.model = inception_v3(weights=weights, aux_logits=True)
            # Modify main classifier
            in_features = self.model.fc.in_features
            self.model.fc = nn.Linear(in_features, num_classes)
            # Modify auxiliary classifier
            if self.model.aux_logits:
                in_features_aux = self.model.AuxLogits.fc.in_features
                self.model.AuxLogits.fc = nn.Linear(in_features_aux, num_classes)
        else:
            raise ValueError(f"Unknown model: {model_name}")
        
        # Optionally freeze backbone
        if freeze_backbone:
            self._freeze_backbone()
    
    def _freeze_backbone(self):
        """Freeze all layers except the classifier."""
        for name, param in self.model.named_parameters():
            if 'fc' not in name and 'classifier' not in name:
                param.requires_grad = False
    
    def unfreeze_all(self):
        """Unfreeze all layers for fine-tuning."""
        for param in self.model.parameters():
            param.requires_grad = True
    
    def forward(self, x):
        return self.model(x)
    
    def get_num_params(self):
        """Return total and trainable parameters."""
        total = sum(p.numel() for p in self.model.parameters())
        trainable = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
        return total, trainable

In [ ]:
# Create model instances
model_configs = {
    'VGG19': 'vgg19',
    'ResNet50': 'resnet50', 
    'GoogLeNet': 'googlenet',
    'InceptionV3': 'inception_v3'
}

# Initialize models
models_dict = {}
for name, model_key in model_configs.items():
    models_dict[name] = TransferLearningModel(
        model_name=model_key,
        num_classes=n_classes,
        pretrained=True,
        freeze_backbone=False  # Fine-tune all layers
    ).to(device)
    
    total, trainable = models_dict[name].get_num_params()
    print(f"{name}: Total params: {total:,} | Trainable: {trainable:,}")

---
## 8. Loss Function & Optimizers

Define the loss function and optimizers for each model.

In [ ]:
# Calculate class weights for imbalanced dataset
def calculate_class_weights(distribution):
    """Calculate class weights inversely proportional to class frequencies."""
    total = distribution.sum()
    weights = total / (len(distribution) * distribution)
    # Normalize weights
    weights = weights / weights.sum() * len(weights)
    return torch.FloatTensor(weights)

class_weights = calculate_class_weights(train_dist)
print("Class weights:")
for i, (name, weight) in enumerate(zip(class_names, class_weights)):
    print(f"  {name}: {weight:.4f}")

In [ ]:
# Loss function with class weights
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))

# Create optimizers for each model
optimizers = {}
schedulers = {}

for name, model in models_dict.items():
    # Adam optimizer with weight decay
    optimizers[name] = optim.AdamW(
        model.parameters(),
        lr=Config.LEARNING_RATE,
        weight_decay=Config.WEIGHT_DECAY
    )
    
    # Learning rate scheduler - ReduceLROnPlateau
    schedulers[name] = optim.lr_scheduler.ReduceLROnPlateau(
        optimizers[name],
        mode='min',
        factor=0.5,
        patience=3,
        verbose=True
    )

print(f"Loss function: CrossEntropyLoss (with class weights)")
print(f"Optimizer: AdamW (lr={Config.LEARNING_RATE}, weight_decay={Config.WEIGHT_DECAY})")
print(f"Scheduler: ReduceLROnPlateau (factor=0.5, patience=3)")

---
## 9. Training & Validation Loops

Define training and validation functions.

In [ ]:
def train_one_epoch(model, train_loader, criterion, optimizer, device, model_name):
    """Train the model for one epoch."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(train_loader, desc='Training', leave=False)
    for images, labels in pbar:
        images = images.to(device)
        labels = labels.squeeze().long().to(device)
        
        # Zero gradients
        optimizer.zero_grad()
        
        # Forward pass
        if model_name in ['GoogLeNet', 'InceptionV3'] and model.training:
            # Handle auxiliary outputs
            outputs = model(images)
            if isinstance(outputs, tuple):
                # GoogLeNet returns (main, aux1, aux2) or InceptionV3 returns InceptionOutputs
                if hasattr(outputs, 'logits'):
                    # InceptionV3
                    loss = criterion(outputs.logits, labels)
                    if outputs.aux_logits is not None:
                        loss += 0.3 * criterion(outputs.aux_logits, labels)
                    outputs = outputs.logits
                else:
                    # GoogLeNet
                    loss = criterion(outputs[0], labels)
                    if len(outputs) > 1 and outputs[1] is not None:
                        loss += 0.3 * criterion(outputs[1], labels)
                    if len(outputs) > 2 and outputs[2] is not None:
                        loss += 0.3 * criterion(outputs[2], labels)
                    outputs = outputs[0]
            else:
                loss = criterion(outputs, labels)
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)
        
        # Backward pass
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        # Statistics
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        pbar.set_postfix({'loss': loss.item(), 'acc': 100. * correct / total})
    
    epoch_loss = running_loss / total
    epoch_acc = 100. * correct / total
    
    return epoch_loss, epoch_acc

In [ ]:
def validate(model, val_loader, criterion, device):
    """Validate the model."""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        pbar = tqdm(val_loader, desc='Validating', leave=False)
        for images, labels in pbar:
            images = images.to(device)
            labels = labels.squeeze().long().to(device)
            
            outputs = model(images)
            
            # Handle InceptionV3 output format during eval
            if hasattr(outputs, 'logits'):
                outputs = outputs.logits
            elif isinstance(outputs, tuple):
                outputs = outputs[0]
            
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            probs = F.softmax(outputs, dim=1)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    
    epoch_loss = running_loss / total
    epoch_acc = 100. * correct / total
    
    return epoch_loss, epoch_acc, np.array(all_preds), np.array(all_labels), np.array(all_probs)

In [ ]:
def train_model(model, model_name, train_loader, val_loader, criterion, optimizer, scheduler, 
                num_epochs, device, checkpoint_dir):
    """Complete training loop for a model."""
    history = {
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': []
    }
    
    best_val_acc = 0.0
    best_model_weights = copy.deepcopy(model.state_dict())
    
    print(f"\n{'='*60}")
    print(f"Training {model_name}")
    print(f"{'='*60}")
    
    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch + 1}/{num_epochs}")
        print("-" * 40)
        
        # Training
        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, device, model_name
        )
        
        # Validation
        val_loss, val_acc, _, _, _ = validate(model, val_loader, criterion, device)
        
        # Update scheduler
        scheduler.step(val_loss)
        
        # Save history
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
        print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")
        
        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_weights = copy.deepcopy(model.state_dict())
            checkpoint_path = os.path.join(checkpoint_dir, f'{model_name}_best.pth')
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_acc': val_acc,
                'val_loss': val_loss
            }, checkpoint_path)
            print(f"  -> Best model saved! (Val Acc: {val_acc:.2f}%)")
    
    # Load best model weights
    model.load_state_dict(best_model_weights)
    print(f"\nTraining completed! Best validation accuracy: {best_val_acc:.2f}%")
    
    return model, history

---
## 10. Model Training

Train all models and save their training histories.

In [ ]:
# Dictionary to store training histories
all_histories = {}

# Train each model
for name in model_configs.keys():
    # Select appropriate data loaders
    if name == 'InceptionV3':
        train_dl = train_loader_inception
        val_dl = val_loader_inception
    else:
        train_dl = train_loader
        val_dl = val_loader
    
    # Train model
    models_dict[name], history = train_model(
        model=models_dict[name],
        model_name=name,
        train_loader=train_dl,
        val_loader=val_dl,
        criterion=criterion,
        optimizer=optimizers[name],
        scheduler=schedulers[name],
        num_epochs=Config.NUM_EPOCHS,
        device=device,
        checkpoint_dir=Config.CHECKPOINT_DIR
    )
    
    all_histories[name] = history

print("\n" + "="*60)
print("All models trained successfully!")
print("="*60)

---
## 11. Learning Curves

Visualize training progress with loss and accuracy curves.

In [ ]:
def plot_learning_curves(histories, figsize=(16, 10)):
    """Plot learning curves for all models."""
    n_models = len(histories)
    fig, axes = plt.subplots(2, n_models, figsize=figsize)
    
    colors = {'train': '#2196F3', 'val': '#FF5722'}
    
    for idx, (name, history) in enumerate(histories.items()):
        epochs = range(1, len(history['train_loss']) + 1)
        
        # Loss curves
        axes[0, idx].plot(epochs, history['train_loss'], label='Train', color=colors['train'], linewidth=2)
        axes[0, idx].plot(epochs, history['val_loss'], label='Validation', color=colors['val'], linewidth=2)
        axes[0, idx].set_title(f'{name} - Loss', fontsize=12, fontweight='bold')
        axes[0, idx].set_xlabel('Epoch')
        axes[0, idx].set_ylabel('Loss')
        axes[0, idx].legend()
        axes[0, idx].grid(True, alpha=0.3)
        
        # Accuracy curves
        axes[1, idx].plot(epochs, history['train_acc'], label='Train', color=colors['train'], linewidth=2)
        axes[1, idx].plot(epochs, history['val_acc'], label='Validation', color=colors['val'], linewidth=2)
        axes[1, idx].set_title(f'{name} - Accuracy', fontsize=12, fontweight='bold')
        axes[1, idx].set_xlabel('Epoch')
        axes[1, idx].set_ylabel('Accuracy (%)')
        axes[1, idx].legend()
        axes[1, idx].grid(True, alpha=0.3)
    
    plt.suptitle('Learning Curves for All Models', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('learning_curves.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_learning_curves(all_histories)

In [ ]:
def plot_combined_accuracy(histories, figsize=(12, 6)):
    """Plot combined validation accuracy for all models."""
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    
    colors = sns.color_palette("husl", len(histories))
    
    # Combined loss
    for idx, (name, history) in enumerate(histories.items()):
        epochs = range(1, len(history['val_loss']) + 1)
        axes[0].plot(epochs, history['val_loss'], label=name, color=colors[idx], linewidth=2, marker='o', markersize=4)
    
    axes[0].set_title('Validation Loss Comparison', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Combined accuracy
    for idx, (name, history) in enumerate(histories.items()):
        epochs = range(1, len(history['val_acc']) + 1)
        axes[1].plot(epochs, history['val_acc'], label=name, color=colors[idx], linewidth=2, marker='o', markersize=4)
    
    axes[1].set_title('Validation Accuracy Comparison', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy (%)')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('combined_metrics.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_combined_accuracy(all_histories)

---
## 12. Ensemble Methods

Implement ensemble methods: Average Probability and Majority Voting.

In [ ]:
class EnsembleModel:
    """Ensemble model combining multiple classifiers."""
    
    def __init__(self, models_dict, device):
        self.models = models_dict
        self.device = device
        
    def predict_proba(self, images, model_name):
        """Get probability predictions from a single model."""
        model = self.models[model_name]
        model.eval()
        
        with torch.no_grad():
            outputs = model(images)
            if hasattr(outputs, 'logits'):
                outputs = outputs.logits
            elif isinstance(outputs, tuple):
                outputs = outputs[0]
            probs = F.softmax(outputs, dim=1)
        
        return probs
    
    def ensemble_average(self, images_dict):
        """Ensemble by averaging probabilities."""
        all_probs = []
        
        for name, images in images_dict.items():
            probs = self.predict_proba(images, name)
            all_probs.append(probs)
        
        # Average probabilities
        avg_probs = torch.stack(all_probs).mean(dim=0)
        predictions = avg_probs.argmax(dim=1)
        
        return predictions, avg_probs
    
    def ensemble_voting(self, images_dict):
        """Ensemble by majority voting."""
        all_preds = []
        all_probs = []
        
        for name, images in images_dict.items():
            probs = self.predict_proba(images, name)
            preds = probs.argmax(dim=1)
            all_preds.append(preds)
            all_probs.append(probs)
        
        # Stack predictions: [n_models, batch_size]
        stacked_preds = torch.stack(all_preds)
        
        # Majority voting
        votes, _ = torch.mode(stacked_preds, dim=0)
        
        # Average probabilities for confidence
        avg_probs = torch.stack(all_probs).mean(dim=0)
        
        return votes, avg_probs
    
    def ensemble_weighted_average(self, images_dict, weights):
        """Ensemble by weighted average of probabilities."""
        all_probs = []
        weight_tensor = torch.tensor(weights, device=self.device)
        weight_tensor = weight_tensor / weight_tensor.sum()  # Normalize
        
        for name, images in images_dict.items():
            probs = self.predict_proba(images, name)
            all_probs.append(probs)
        
        # Weighted average
        stacked_probs = torch.stack(all_probs)  # [n_models, batch, n_classes]
        weighted_probs = (stacked_probs * weight_tensor.view(-1, 1, 1)).sum(dim=0)
        predictions = weighted_probs.argmax(dim=1)
        
        return predictions, weighted_probs

# Create ensemble model
ensemble = EnsembleModel(models_dict, device)
print("Ensemble model created!")

In [ ]:
def evaluate_ensemble(ensemble, test_loader_standard, test_loader_inception, device, method='average'):
    """Evaluate ensemble model on test set."""
    all_preds = []
    all_labels = []
    all_probs = []
    
    # Create iterators
    standard_iter = iter(test_loader_standard)
    inception_iter = iter(test_loader_inception)
    
    n_batches = min(len(test_loader_standard), len(test_loader_inception))
    
    pbar = tqdm(range(n_batches), desc=f'Evaluating Ensemble ({method})')
    
    for _ in pbar:
        # Get batches
        images_standard, labels = next(standard_iter)
        images_inception, _ = next(inception_iter)
        
        images_standard = images_standard.to(device)
        images_inception = images_inception.to(device)
        labels = labels.squeeze().numpy()
        
        # Prepare images dict
        images_dict = {
            'VGG19': images_standard,
            'ResNet50': images_standard,
            'GoogLeNet': images_standard,
            'InceptionV3': images_inception
        }
        
        # Get ensemble predictions
        if method == 'average':
            preds, probs = ensemble.ensemble_average(images_dict)
        elif method == 'voting':
            preds, probs = ensemble.ensemble_voting(images_dict)
        elif method == 'weighted':
            # Use validation accuracies as weights
            weights = [max(all_histories[name]['val_acc']) for name in images_dict.keys()]
            preds, probs = ensemble.ensemble_weighted_average(images_dict, weights)
        else:
            raise ValueError(f"Unknown method: {method}")
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels)
        all_probs.extend(probs.cpu().numpy())
    
    return np.array(all_preds), np.array(all_labels), np.array(all_probs)

In [ ]:
# Evaluate ensemble methods
ensemble_results = {}

for method in ['average', 'voting', 'weighted']:
    preds, labels, probs = evaluate_ensemble(
        ensemble, test_loader, test_loader_inception, device, method=method
    )
    accuracy = accuracy_score(labels, preds) * 100
    ensemble_results[f'Ensemble ({method})'] = {
        'predictions': preds,
        'labels': labels,
        'probabilities': probs,
        'accuracy': accuracy
    }
    print(f"Ensemble ({method}) Test Accuracy: {accuracy:.2f}%")

---
## 13. Model Evaluation

Evaluate all individual models and ensemble methods on the test set.

In [ ]:
def evaluate_model_on_test(model, test_loader, criterion, device, model_name):
    """Evaluate a single model on the test set."""
    print(f"\nEvaluating {model_name}...")
    test_loss, test_acc, preds, labels, probs = validate(model, test_loader, criterion, device)
    print(f"{model_name} Test Loss: {test_loss:.4f} | Test Accuracy: {test_acc:.2f}%")
    return {
        'predictions': preds,
        'labels': labels,
        'probabilities': probs,
        'accuracy': test_acc,
        'loss': test_loss
    }

# Evaluate individual models
individual_results = {}

for name in model_configs.keys():
    if name == 'InceptionV3':
        test_dl = test_loader_inception
    else:
        test_dl = test_loader
    
    individual_results[name] = evaluate_model_on_test(
        models_dict[name], test_dl, criterion, device, name
    )

In [ ]:
def plot_confusion_matrix(labels, predictions, class_names, title, figsize=(10, 8)):
    """Plot confusion matrix."""
    cm = confusion_matrix(labels, predictions)
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    fig, axes = plt.subplots(1, 2, figsize=(figsize[0] * 2, figsize[1]))
    
    # Raw counts
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names, ax=axes[0])
    axes[0].set_title(f'{title}\n(Counts)', fontsize=12)
    axes[0].set_xlabel('Predicted')
    axes[0].set_ylabel('True')
    
    # Normalized
    sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=axes[1])
    axes[1].set_title(f'{title}\n(Normalized)', fontsize=12)
    axes[1].set_xlabel('Predicted')
    axes[1].set_ylabel('True')
    
    plt.tight_layout()
    plt.savefig(f'confusion_matrix_{title.replace(" ", "_").lower()}.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    return cm

In [ ]:
# Plot confusion matrices for individual models
for name, results in individual_results.items():
    plot_confusion_matrix(
        results['labels'], 
        results['predictions'], 
        class_names, 
        f'{name}'
    )

In [ ]:
# Plot confusion matrices for ensemble methods
for name, results in ensemble_results.items():
    plot_confusion_matrix(
        results['labels'], 
        results['predictions'], 
        class_names, 
        name
    )

In [ ]:
def print_classification_report(labels, predictions, class_names, model_name):
    """Print detailed classification report."""
    print(f"\n{'='*60}")
    print(f"Classification Report: {model_name}")
    print(f"{'='*60}")
    print(classification_report(labels, predictions, target_names=class_names, digits=4))

# Print classification reports for all models
for name, results in individual_results.items():
    print_classification_report(results['labels'], results['predictions'], class_names, name)

for name, results in ensemble_results.items():
    print_classification_report(results['labels'], results['predictions'], class_names, name)

In [ ]:
def plot_roc_curves(labels, probabilities, class_names, title, figsize=(12, 10)):
    """Plot ROC curves for multi-class classification."""
    n_classes = len(class_names)
    
    # Binarize labels
    labels_binary = label_binarize(labels, classes=list(range(n_classes)))
    
    # Compute ROC curve and AUC for each class
    fpr = {}
    tpr = {}
    roc_auc = {}
    
    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(labels_binary[:, i], probabilities[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
    
    # Plot
    fig, ax = plt.subplots(figsize=figsize)
    colors = sns.color_palette("husl", n_classes)
    
    for i in range(n_classes):
        ax.plot(fpr[i], tpr[i], color=colors[i], linewidth=2,
                label=f'{class_names[i]} (AUC = {roc_auc[i]:.3f})')
    
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel('False Positive Rate', fontsize=12)
    ax.set_ylabel('True Positive Rate', fontsize=12)
    ax.set_title(f'ROC Curves - {title}', fontsize=14, fontweight='bold')
    ax.legend(loc='lower right', fontsize=10)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'roc_curves_{title.replace(" ", "_").lower()}.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Calculate macro-average AUC
    macro_auc = np.mean(list(roc_auc.values()))
    print(f"Macro-average AUC: {macro_auc:.4f}")
    
    return roc_auc

In [ ]:
# Plot ROC curves for individual models and ensemble
all_aucs = {}

for name, results in individual_results.items():
    all_aucs[name] = plot_roc_curves(
        results['labels'], 
        results['probabilities'], 
        class_names, 
        name
    )

# Best ensemble method
best_ensemble = max(ensemble_results.items(), key=lambda x: x[1]['accuracy'])
all_aucs[best_ensemble[0]] = plot_roc_curves(
    best_ensemble[1]['labels'],
    best_ensemble[1]['probabilities'],
    class_names,
    best_ensemble[0]
)

---
## 14. Results Comparison

Compare performance of all models and ensemble methods.

In [ ]:
# Compile results
results_summary = []

# Individual models
for name, results in individual_results.items():
    precision, recall, f1, _ = precision_recall_fscore_support(
        results['labels'], results['predictions'], average='macro'
    )
    results_summary.append({
        'Model': name,
        'Type': 'Individual',
        'Accuracy (%)': results['accuracy'],
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'Macro AUC': np.mean(list(all_aucs.get(name, {i: 0 for i in range(n_classes)}).values()))
    })

# Ensemble models
for name, results in ensemble_results.items():
    precision, recall, f1, _ = precision_recall_fscore_support(
        results['labels'], results['predictions'], average='macro'
    )
    results_summary.append({
        'Model': name,
        'Type': 'Ensemble',
        'Accuracy (%)': results['accuracy'],
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'Macro AUC': np.mean(list(all_aucs.get(name, {i: 0 for i in range(n_classes)}).values()))
    })

# Create DataFrame
results_df = pd.DataFrame(results_summary)
results_df = results_df.sort_values('Accuracy (%)', ascending=False)

print("\n" + "="*80)
print("PERFORMANCE COMPARISON - ALL MODELS")
print("="*80)
print(results_df.to_string(index=False))

In [ ]:
# Visualize performance comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Prepare data
models = results_df['Model'].tolist()
colors = ['#2196F3' if t == 'Individual' else '#4CAF50' for t in results_df['Type']]

# Accuracy comparison
bars1 = axes[0].barh(models, results_df['Accuracy (%)'], color=colors)
axes[0].set_xlabel('Accuracy (%)')
axes[0].set_title('Test Accuracy Comparison', fontsize=12, fontweight='bold')
axes[0].axvline(x=results_df['Accuracy (%)'].mean(), color='red', linestyle='--', label='Mean')
for bar, acc in zip(bars1, results_df['Accuracy (%)']):
    axes[0].text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2, 
                 f'{acc:.1f}%', va='center', fontsize=9)

# F1-Score comparison
bars2 = axes[1].barh(models, results_df['F1-Score'], color=colors)
axes[1].set_xlabel('F1-Score')
axes[1].set_title('F1-Score Comparison', fontsize=12, fontweight='bold')
for bar, f1 in zip(bars2, results_df['F1-Score']):
    axes[1].text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                 f'{f1:.3f}', va='center', fontsize=9)

# Model type legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#2196F3', label='Individual'),
                   Patch(facecolor='#4CAF50', label='Ensemble')]

# Best validation accuracy per model
best_val_accs = [max(all_histories[name]['val_acc']) if name in all_histories else 0 
                 for name in models]
test_accs = results_df['Accuracy (%)'].tolist()

x = np.arange(len([m for m in models if m in all_histories]))
width = 0.35
individual_models = [m for m in models if m in all_histories]
individual_val = [max(all_histories[m]['val_acc']) for m in individual_models]
individual_test = [results_df[results_df['Model'] == m]['Accuracy (%)'].values[0] for m in individual_models]

axes[2].bar(x - width/2, individual_val, width, label='Validation', color='#2196F3')
axes[2].bar(x + width/2, individual_test, width, label='Test', color='#FF5722')
axes[2].set_xlabel('Model')
axes[2].set_ylabel('Accuracy (%)')
axes[2].set_title('Validation vs Test Accuracy', fontsize=12, fontweight='bold')
axes[2].set_xticks(x)
axes[2].set_xticklabels(individual_models, rotation=45, ha='right')
axes[2].legend()

plt.tight_layout()
plt.savefig('performance_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Per-class performance comparison
def get_per_class_metrics(labels, predictions, class_names):
    """Get per-class precision, recall, and F1-score."""
    precision, recall, f1, support = precision_recall_fscore_support(
        labels, predictions, average=None
    )
    return pd.DataFrame({
        'Class': class_names,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'Support': support
    })

# Compare per-class performance
print("\n" + "="*80)
print("PER-CLASS PERFORMANCE COMPARISON")
print("="*80)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

best_individual = max(individual_results.items(), key=lambda x: x[1]['accuracy'])
best_ensemble_item = max(ensemble_results.items(), key=lambda x: x[1]['accuracy'])

comparison_models = [
    ('VGG19', individual_results['VGG19']),
    ('ResNet50', individual_results['ResNet50']),
    (best_individual[0], best_individual[1]),
    (best_ensemble_item[0], best_ensemble_item[1])
]

for idx, (name, results) in enumerate(comparison_models):
    metrics_df = get_per_class_metrics(results['labels'], results['predictions'], class_names)
    
    x = np.arange(len(class_names))
    width = 0.25
    
    axes[idx].bar(x - width, metrics_df['Precision'], width, label='Precision', color='#2196F3')
    axes[idx].bar(x, metrics_df['Recall'], width, label='Recall', color='#4CAF50')
    axes[idx].bar(x + width, metrics_df['F1-Score'], width, label='F1-Score', color='#FF5722')
    
    axes[idx].set_xlabel('Class')
    axes[idx].set_ylabel('Score')
    axes[idx].set_title(f'{name}', fontsize=11, fontweight='bold')
    axes[idx].set_xticks(x)
    axes[idx].set_xticklabels([f'{i}' for i in range(len(class_names))], rotation=0)
    axes[idx].legend(fontsize=8)
    axes[idx].set_ylim(0, 1.1)
    axes[idx].grid(True, alpha=0.3, axis='y')

plt.suptitle('Per-Class Performance Metrics', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('per_class_performance.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 15. Conclusion

Summary of findings and model recommendations.

In [ ]:
# Final Summary
print("\n" + "="*80)
print("MEDICAL IMAGE CLASSIFICATION - FINAL SUMMARY")
print("="*80)

print(f"\nDataset: {Config.DATA_FLAG.upper()} (DermaMNIST)")
print(f"Task: {n_classes}-class classification of skin lesions")
print(f"Training samples: {len(train_dataset_raw)}")
print(f"Test samples: {len(test_dataset_raw)}")

print("\n" + "-"*60)
print("INDIVIDUAL MODEL PERFORMANCE (Test Set)")
print("-"*60)
for name in model_configs.keys():
    acc = individual_results[name]['accuracy']
    print(f"  {name:15s}: {acc:.2f}%")

best_individual_name = max(individual_results.items(), key=lambda x: x[1]['accuracy'])[0]
best_individual_acc = individual_results[best_individual_name]['accuracy']
print(f"\n  Best Individual Model: {best_individual_name} ({best_individual_acc:.2f}%)")

print("\n" + "-"*60)
print("ENSEMBLE MODEL PERFORMANCE (Test Set)")
print("-"*60)
for name, results in ensemble_results.items():
    print(f"  {name:25s}: {results['accuracy']:.2f}%")

best_ensemble_name = max(ensemble_results.items(), key=lambda x: x[1]['accuracy'])[0]
best_ensemble_acc = ensemble_results[best_ensemble_name]['accuracy']
print(f"\n  Best Ensemble Method: {best_ensemble_name} ({best_ensemble_acc:.2f}%)")

# Improvement analysis
improvement = best_ensemble_acc - best_individual_acc
print("\n" + "-"*60)
print("ENSEMBLE IMPROVEMENT ANALYSIS")
print("-"*60)
if improvement > 0:
    print(f"  Ensemble improvement over best individual: +{improvement:.2f}%")
else:
    print(f"  Best individual model outperforms ensemble by: {-improvement:.2f}%")

print("\n" + "-"*60)
print("KEY OBSERVATIONS")
print("-"*60)
print("  1. Transfer learning with pretrained ImageNet models is effective")
print("     for medical image classification.")
print("  2. Data augmentation helps improve generalization on limited")
print("     medical imaging data.")
print("  3. Class imbalance significantly affects minority class performance.")
print("  4. Ensemble methods can help reduce variance in predictions.")

print("\n" + "="*80)
print("TRAINING COMPLETE!")
print("="*80)

In [ ]:
# Save results to CSV
results_df.to_csv('model_performance_results.csv', index=False)
print("Results saved to 'model_performance_results.csv'")

# Save training histories
import json

histories_serializable = {}
for name, history in all_histories.items():
    histories_serializable[name] = {
        k: [float(v) for v in vals] for k, vals in history.items()
    }

with open('training_histories.json', 'w') as f:
    json.dump(histories_serializable, f, indent=2)
print("Training histories saved to 'training_histories.json'")

In [ ]:
# Inference example - Predict on a single image
def predict_single_image(image_path_or_tensor, models_dict, ensemble, device, class_names):
    """Make predictions on a single image using all models and ensemble."""
    
    # If it's a path, load and transform the image
    if isinstance(image_path_or_tensor, str):
        image = Image.open(image_path_or_tensor).convert('RGB')
        img_standard = val_transform(image).unsqueeze(0).to(device)
        img_inception = val_transform_inception(image).unsqueeze(0).to(device)
    else:
        img_standard = image_path_or_tensor.unsqueeze(0).to(device) if image_path_or_tensor.dim() == 3 else image_path_or_tensor.to(device)
        # For inception, we need to resize
        img_inception = F.interpolate(img_standard, size=(299, 299), mode='bilinear', align_corners=False)
    
    results = {}
    
    # Individual models
    for name, model in models_dict.items():
        model.eval()
        with torch.no_grad():
            if name == 'InceptionV3':
                outputs = model(img_inception)
            else:
                outputs = model(img_standard)
            
            if hasattr(outputs, 'logits'):
                outputs = outputs.logits
            elif isinstance(outputs, tuple):
                outputs = outputs[0]
            
            probs = F.softmax(outputs, dim=1)
            pred = probs.argmax(dim=1).item()
            conf = probs[0, pred].item()
            
            results[name] = {
                'prediction': class_names[pred],
                'confidence': conf,
                'class_idx': pred
            }
    
    # Ensemble prediction
    images_dict = {
        'VGG19': img_standard,
        'ResNet50': img_standard,
        'GoogLeNet': img_standard,
        'InceptionV3': img_inception
    }
    
    ens_pred, ens_probs = ensemble.ensemble_average(images_dict)
    ens_class = ens_pred.item()
    ens_conf = ens_probs[0, ens_class].item()
    
    results['Ensemble'] = {
        'prediction': class_names[ens_class],
        'confidence': ens_conf,
        'class_idx': ens_class
    }
    
    return results

# Demo: Predict on a test sample
test_img, test_label = test_dataset_raw[0]
predictions = predict_single_image(
    val_transform(test_img), 
    models_dict, 
    ensemble, 
    device, 
    class_names
)

print("\nPrediction Example:")
print(f"True Label: {class_names[test_label[0]]}")
print("-" * 40)
for model_name, result in predictions.items():
    print(f"{model_name:15s}: {result['prediction']:20s} (conf: {result['confidence']:.2%})")